# Touch and sight referents

Section 5.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

import pandas as pd

# (verb, argument) rows from BookNLP, built by embodiment/Analysis/booknlp_touch_referents.ipynb
REFERENTS = cfg.ANNOTATION_DIR / "Data" / "booknlp_touch_referents.csv"

LABELS = ["haptic_human_animal", "haptic_material",
          "sight_human", "sight_material"]
VERB_SS = {"haptic_human_animal": "verb.contact", "haptic_material": "verb.contact",
           "sight_human": "verb.perception", "sight_material": "verb.perception"}

ref = pd.read_csv(REFERENTS, keep_default_na=False,
                  usecols=["label", "dep", "verb_ss", "obj_lemma", "obj_ss"])
print(f"{len(ref):,} rows")

## Direct objects

In [ ]:
# Direct objects of contact verbs (touch) and perception verbs (sight)
scoped = ref[ref["dep"].isin(["obj", "dobj"]) & (ref["obj_ss"] != "")
             & (ref["verb_ss"] == ref["label"].map(VERB_SS))]

rows = []
for label in LABELS:
    g = scoped[scoped["label"] == label]
    c, n = g["obj_ss"].value_counts(), len(g)
    rows.append({"label": label, "n": n,
                 "person %": c.get("noun.person", 0) / n * 100,
                 "body %": c.get("noun.body", 0) / n * 100,
                 "animal %": c.get("noun.animal", 0) / n * 100})
pd.DataFrame(rows).set_index("label").round(1)

## Person versus animal

In [ ]:
rows = []
for label in LABELS:
    g = scoped[scoped["label"] == label]
    person = int((g["obj_ss"] == "noun.person").sum())
    animal = int((g["obj_ss"] == "noun.animal").sum())
    rows.append({"label": label, "person": person, "animal": animal,
                 "animal share %": animal / (person + animal) * 100})
pd.DataFrame(rows).set_index("label").round(1)

## Animals named

In [ ]:
for label in LABELS:
    g = scoped[(scoped["label"] == label) & (scoped["obj_ss"] == "noun.animal")]
    top = g["obj_lemma"].str.lower().value_counts().head(12)
    print(f"{label}: " + ", ".join(f"{w} ({c:,})" for w, c in top.items()))